# Session 17 · Prompt evaluation
## A golden set, a rubric, and a judge

**How to use this notebook.** It has three parts.

| Part | What it is | Code? |
|---|---|---|
| **Part A · Read before you run** | Why this topic matters, the words you need, and how today's session is built | No code. Read it first (about 15 minutes). |
| **Part B · Hands-on** | Sections 1 to 9. Every section follows the same six steps (below). | Yes |
| **Part C · Wrap-up** | Summary, interview questions, troubleshooting | No |

Every section in Part B follows the same six steps, so you always know where you are:

1. **What and why** — what we are about to do and why it matters.
2. **The idea** — the concept, with an analogy and often a small diagram.
3. **What the next cell does** — a numbered walk-through *before* you see the code.
4. **The code** — run it.
5. **What you should see** — how to read the output, and what it means if yours is different.
6. **Check yourself** — one short question and a common mistake to avoid.

Boxes marked **▸ Read more** are optional. Click to open them when you want the deeper background; you can skip them in class.

---
# Part A · Read before you run

## A1. Why this session matters

### The problem, in one story
Session 16 gave you a number: your structured classifier got 18 of 20 held-out tickets right, your Session 14 baseline got 13. You committed the report and moved on. But a teammate asks a fair question: *"18 out of 20 on what, exactly? The LABEL? What about the reply itself — is it polite, does it invent a refund policy nobody approved, does it follow the format support actually needs?"*

A label is one bit of information. A customer-support reply is judged on several axes at once, and "is the label right" is only one of them. You cannot eyeball a hundred replies across two prompt versions and trust your own memory of which one felt better — that is exactly the kind of ungoverned, undocumented judgement Session 16 moved you away from when it put the prompt in a versioned file.

Today you build the other half of a governed prompt pipeline: a **golden set** (a fixed, agreed list of test cases), a **rubric** (pass/fail checks decided before you read a single answer), and an **LLM-as-judge** (a second model call that grades the first model's reply against the rubric). You already have the golden set — it is Session 16's held-out tickets, reused, not rebuilt.

### What you learn today, and what each one prevents

| You learn to... | It prevents... | The everyday name |
|---|---|---|
| Build a **rubric** before reading any answers | Grading toward whichever answer you already liked | Decide the exam's marking scheme before the exam |
| Use a second model as a **judge**, with strict JSON output | A free-text verdict ("mostly fine, I think") that no script can total up | A second opinion, in a form you can count |
| Name the judge's known **biases** (position, verbosity, self-preference) | Trusting a judge blindly because it sounds confident | Know your reviewer's blind spots |
| **Shuffle** which prompt is judged first, per ticket | A timing or caching quirk quietly favouring one prompt | A fair coin toss, not a fixed running order |
| **Spot-check** five tickets by hand | A judge that nobody has ever checked against a human | An audit sample, not blind trust |

### Where this is used in industry
- **Banking (BFS):** a regulator or an internal model-risk team routinely asks "how do you know this prompt is better than the last one?" A committed golden set, rubric and report is the auditable answer.
- **Healthcare:** summarisation and triage prompts are evaluated against a clinician-reviewed rubric before every rollout, never just "it looked right to the engineer".
- **Insurance and retail:** A/B-testing two prompt versions on live traffic is expensive and slow; an offline judge on a golden set catches most regressions first, cheaply.
- **Any team running more than one prompt version:** without a rubric, "which prompt is better" quietly becomes "which prompt does the loudest person in the room prefer".

### Why interviewers care
Interviewers ask about this because "how do you know your prompt change is actually an improvement?" separates engineers who ship on vibes from engineers who ship on evidence. Be ready for:
1. "How would you evaluate whether a new prompt is better than the old one?"
2. "What is LLM-as-judge, and what are its known failure modes?"
3. "Why would you shuffle answer order in a judge call?"
4. "What is the difference between a golden set and a held-out set?" (Short answer today: for us, none — the held-out set from Session 16 *is* the golden set. The name changes because its job changes, not its contents.)
5. "Would you ever use a human to check an LLM judge, and how?"

Each is answered, with an example from this lab, in Part C.

<details><summary><b>▸ Read more: why not just read the twenty replies yourself?</b></summary>

You could, for twenty tickets and two prompts (forty replies). Real evaluation runs are often hundreds of items across several prompt versions, re-run every time the prompt, the model, or the data changes. Reading all of it by hand does not scale, and a tired human grades the fifth prompt version more leniently than the first. The judge does not get tired, but it has its own biases — section 9 and 10 name them, and the guardrails in section 11 are how real teams keep using it anyway.
</details>

## A2. Where this session fits in the course

```
 S14  Prompt anatomy ──────────► built the support classifier with ONE line of instruction
  │                              (that one line is today's "v1")
 S15  Reasoning prompts ───────► DiskCache (free re-runs) and Meter (token counts) — reused today
  │
 S16  Structured prompting ────► tags + template + prompt file; held-out accuracy (today's "v2")
  │
 S17  Prompt evaluation ───────► TODAY: golden set, rubric, LLM judge — v1 vs v2, scored properly
  │
 S18  Defensive prompting ─────► attacks the Session 16 prompt on purpose; re-measured with THIS judge
 S19  Chat architecture ───────► system prompt as the contract of a full chat assistant
 S53–54  Prompt versioning, LLMOps ► today's golden set + rubric, at team scale and on a schedule
```

**What you need from earlier sessions:** a working `.env` with your Groq key (Session 11), the course repository pulled and synced, and your Session 16 files still in your **work repository**: `prompts/support_classifier/system.md` and `user.j2`. Today does not change them — it grades them.

## A3. The words you need today

Read these once now. Each term appears in bold the first time it is used in Part B.

---
**1. Golden set**
- **Definition:** a fixed, agreed list of test cases, kept exactly as-is across every prompt you later compare.
- **Analogy:** the official answer key for an exam — written once, not edited to fit whoever is sitting the exam.
- **Example:** today's `GOLDEN20` is *exactly* Session 16's `HELDOUT` — the same twenty tickets, same order, same labels.
- **Often confused with:** a held-out set. They can be the same data; the name changes with the job. "Held-out" means *not used as a training/few-shot example*. "Golden" means *the fixed set we score every version against*.
- **Comes back in:** S18, S53–54.

---
**2. Rubric**
- **Definition:** a written list of pass/fail checks, agreed **before** you read any answers.
- **Analogy:** a marking scheme a teacher writes before the exam, not after seeing who wrote what.
- **Example:** today's four criteria — correct label, cites the right house rule, follows the reply format, invents no policy.
- **Often confused with:** a 1-to-10 quality score. A score invites "it felt like an 8"; a rubric forces a yes/no per concrete check.
- **Comes back in:** S53–54.

---
**3. LLM-as-judge**
- **Definition:** using a second model call to grade the first model's output against a rubric.
- **Analogy:** a second pair of eyes marking the same exam with the same marking scheme.
- **Example:** today's `judge_answer()` sends one ticket, one answer and the rubric to the model and asks for PASS/FAIL per criterion.
- **Often confused with:** the model grading *itself* in the same call. Today's answerer and judge are separate calls (even when they happen to be the same underlying model).
- **Comes back in:** S18 (re-measuring after an attack).

---
**4. Strict-JSON verdict**
- **Definition:** forcing the judge's reply into a fixed shape (exactly four named keys, each PASS or FAIL) so code can total it up.
- **Analogy:** a bubble-sheet answer form instead of an essay — harder to mis-read, easy to machine-score.
- **Example:** `{"c1": "PASS", "c2": "FAIL", "c3": "PASS", "c4": "PASS"}`
- **Often confused with:** asking for a written explanation as well. You can ask for one, but never let the score depend on parsing free text.
- **Comes back in:** S20 (function calling asks for structure the same way, for a different reason).

---
**5. Judge bias — position bias**
- **Definition:** a judge's tendency to prefer whichever answer it reads first (or in a fixed, predictable slot), regardless of quality.
- **Analogy:** a recruiter who, without noticing, rates the first candidate of the day more kindly.
- **Example:** always answering v1 before v2, for every ticket, could make v1 look stronger for no reason but order.
- **Often confused with:** a real quality difference. The fix (section 11) is to vary the order, then see if the gap survives.
- **Comes back in:** S18.

---
**6. Judge bias — verbosity bias**
- **Definition:** a judge's tendency to reward longer, more confident-sounding replies, even when a short one is objectively correct.
- **Analogy:** grading an exam answer higher just because it filled the page.
- **Example:** a padded reply that still gets the label wrong scoring better than a terse, correct one.
- **Often confused with:** thoroughness actually being asked for. The fix is a pass/fail rubric rather than an open "rate the quality" scale, which resists this better (not perfectly).
- **Comes back in:** —

---
**7. Judge bias — self-preference bias**
- **Definition:** a model rating output that resembles its own typical phrasing or reasoning style more highly.
- **Analogy:** a teacher who was once a journalist marking essays written in journalistic style more kindly.
- **Example:** today's answerer and judge are the same model family — a real limit of this lab, named honestly rather than hidden.
- **Often confused with:** a factual quality judgement. The fix named on the guardrails slide — a different, ideally stronger, model as judge — is not exercised in this notebook; it is a known next step.
- **Comes back in:** —

---
**8. Order guard**
- **Definition:** deterministically varying, per test item, which of two things happens first, so no fixed ordering can bias the result.
- **Analogy:** alternating who serves first in a match, rather than always the same player.
- **Example:** `shuffle_order()` decides per ticket whether v1 or v2 is answered and judged first — the same choice every run (no hidden randomness), but not always the same prompt.
- **Often confused with:** true randomness. It is a deterministic hash, so your report is reproducible.
- **Comes back in:** —

---
**9. Human spot-check**
- **Definition:** grading a small sample yourself, with the same rubric, and comparing with the judge's verdicts.
- **Analogy:** a teacher re-marking five exam papers from a batch a teaching assistant marked, to check the TA's marking is sound.
- **Example:** today you grade five tickets by hand and compute an agreement rate with the judge.
- **Often confused with:** proof the judge is right everywhere. Five tickets is evidence, not a guarantee, for every ticket the judge ever grades.
- **Comes back in:** —

---
**10. Pass rate**
- **Definition:** the fraction of graded items that passed a given criterion (or passed every criterion at once).
- **Analogy:** the percentage of students who passed a particular exam question.
- **Example:** `16/20 = 80%` for "correct label".
- **Often confused with:** accuracy in the Session 16 sense. Session 16's accuracy was one criterion (the label) measured directly. Today's pass rate is one of *four* rubric criteria, each graded by the judge, not by exact string comparison.
- **Comes back in:** S53–54.

## A4. How this session is built

### The big picture

```
┌──────────────────────────── COURSE REPOSITORY (you pull it; you do not edit it) ───────────────────────────┐
│                                                                                                            │
│  module02/structured_tasks.py (S16)   HELDOUT = 20 labelled tickets, 5 per label                           │
│     evaluation_tasks.py (today)   THE DATA                                                                  │
│     GOLDEN20 = structured_tasks.HELDOUT, reused, not rebuilt       RUBRIC = 4 pass/fail criteria             │
│     V1_INSTRUCTION = Session 14's baseline        JUDGE_SYSTEM / JUDGE_USER = the judge's own prompt         │
│                                                                                                            │
│  module02/evaluation_utils.py (today)   THE TOOLS (helpers this notebook calls)                             │
│     judge_messages · parse_verdict · judge_answer · score_prompt · compare_prompts · shuffle_order ·        │
│     spot_check_items · agreement · eval_markdown · check_s17_work                                           │
│                                                                                                            │
│  module02/structured_utils.py (S16)   load_prompt_file · pick_examples · structured_maker — v2 is built     │
│                                        from YOUR files using these same Session 16 helpers                  │
│  module02/reasoning_utils.py (S15)    Lab = talks to the model · DiskCache = free re-runs · Meter = tokens  │
│  module01/providers_utils.py (S11)    reads your .env and makes the Groq client                             │
│                                                                                                            │
│  module02/s17_evaluation.ipynb  ◄── YOU ARE HERE                                                            │
│  tests/test_s17.py               the checkpoint (pytest)                                                    │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
                                   │ the notebook reads your S16 files, writes a new report ▼
┌──────────────────────────── YOUR WORK REPOSITORY (yours; you commit and push it) ──────────────────────────┐
│  prompts/support_classifier/system.md   YOUR Session 16 file — read today, not changed                      │
│  prompts/support_classifier/user.j2     YOUR Session 16 file — read today, not changed                      │
│  experiments/s17_eval.md                NEW today: pass rates for v1 and v2, plus your spot-check            │
└────────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### The journey of one ticket, today
```
 one GOLDEN20 ticket
        │
        ├──► answered by v1 (S14 baseline) ──► judged against the rubric ──► 4 x PASS/FAIL
        │
        └──► answered by v2 (YOUR S16 files) ──► judged against the rubric ──► 4 x PASS/FAIL
                                                       │
                                          which answer is judged FIRST is shuffled, per ticket
```

### Every helper this notebook calls

| Helper | Lives in | You give it | It gives back | Used in section |
|---|---|---|---|---|
| `v1_messages` | `evaluation_utils` | a ticket | messages for the S14 baseline | 5 |
| `v2_messages_from_files` | `evaluation_utils` | your work repo, a ticket | messages built from YOUR system.md + user.j2 | 6 |
| `answer_with` | `evaluation_utils` | a Lab, messages | the model's reply text | 5, 6 |
| `judge_messages` | `evaluation_utils` | ticket, truth, answer | messages for the judge | 4 |
| `parse_verdict` | `evaluation_utils` | the judge's reply text | `{"c1": True/False, ...}` | 4 |
| `judge_answer` | `evaluation_utils` | a Lab, ticket, truth, answer | one graded verdict | 4, 5, 6 |
| `shuffle_order` | `evaluation_utils` | the golden set | which prompt goes first, per ticket | 6 |
| `score_prompt` | `evaluation_utils` | answer Lab, judge Lab, a messages-builder | pass rates for one prompt | 5 |
| `compare_prompts` | `evaluation_utils` | both Labs, both messages-builders | pass rates for v1 AND v2, order-guarded | 6 |
| `spot_check_items` | `evaluation_utils` | the golden set | 5 tickets, deterministic | 7 |
| `agreement` / `agreement_rate` | `evaluation_utils` | human and judge verdicts | how often they agree | 7 |
| `eval_markdown` | `evaluation_utils` | both results, models, spot-check | the report text | 8 |
| `save_eval_report` / `check_s17_work` | `evaluation_utils` | your work repo | save the report / check your submission | 8 |

### The order of the sections

| # | Section | Calls the model? | Minutes |
|---|---|---|---|
| 1 | Setup | No | 3 |
| 2 | Predict first: two replies, which is better? | No | 5 |
| 3 | The golden set and the rubric | No | 4 |
| 4 | Write and test the judge on one ticket | Yes (1 answer + 1 judge call) | 6 |
| 5 | Score v1 (baseline) on all 20 | Yes (20 + 20 calls) | 7 |
| 6 | Score v2 (structured) on all 20, compare | Yes (20 + 20 calls) | 8 |
| 7 | Human spot-check (5 tickets) | No (uses section 5–6's saved rows) | 6 |
| 8 | Save your evaluation report, check your submission | No | 4 |
| 9 | Commit, and run the checkpoint | No (shell + pytest) | 3 |

## A5. What you will be able to do, and what you will submit

**By the end of this session you will be able to:**
1. Explain why a label match is not the same as a good reply, with a concrete example.
2. Write a rubric of pass/fail criteria before reading any answers.
3. Build a judge prompt that returns strict, parseable JSON, and parse it defensively.
4. Compare two prompt versions on the same golden set with the same judge, guarded against ordering bias.
5. Name three known biases of LLM-as-judge and the guardrail for each.
6. Grade a small sample by hand and compute agreement with the judge.

**What you submit (in your work repository):**

| File | What it must contain |
|---|---|
| `prompts/support_classifier/system.md`, `user.j2` | Unchanged from Session 16 — today reads them, does not edit them |
| `experiments/s17_eval.md` | Pass rates for v1 AND v2 across all four criteria, all twenty golden tickets, plus a five-ticket human spot-check |

**The checkpoint:** `uv run pytest tests/test_s17.py -rs` with `WORK_REPO` set. *A skipped test is not a pass.*

**Budget:** today makes roughly twice as many model calls as Session 16, because every ticket is answered AND judged, for both prompt versions: about 80 calls (20 tickets x 2 prompts x 2 — one answer call, one judge call). Expect **60,000–90,000** of your 200,000 daily tokens (an estimate; the meter shows your real number). Re-running an unchanged cell is free.

---
# Part B · Hands-on

## 1. Setup

### What and why
Same pattern as every session: find the course files, read your `.env`, connect to the model, and open today's cache and meter. If something fails, it should fail now, loudly, not halfway through section 6.

### The idea
Opening the branch in the morning: unlock the door (find the files), switch on the system (read your key), check the counter is ready (connect to the model). If the system is down, you can still do the paperwork — sections 2 and 3 need no model call at all.

### What the next cell does
1. `sys.path[:0] = [...]` adds the course folders to the front of Python's search list, so the imports work whether you opened the notebook from the repo root or from `module02/`.
2. The `try / except ImportError` block loads helpers from Sessions 11, 14, 15, 16 and today's own two files, and turns a missing-file error into one that tells you what to do.
3. `groq_client()` (Session 11) already called `load_dotenv(override=True)`, so `os.getenv(...)` sees your `.env`.
4. `DiskCache(S17_CACHE)` opens *today's own* cache file — a fresh one from Session 16's, because today's calls (answer + judge) are different calls with different prompts.
5. `WORK_REPO` is read from the environment: this is where section 6 will load YOUR Session 16 files from.

### The code

In [ ]:
import sys; sys.path[:0] = [".", "module01", "module02", "../module01", "../module02"]
import os
from pathlib import Path
try:
    from providers_utils import groq_client                                              # Session 11
    from reasoning_utils import DiskCache, Lab, Meter                                     # Session 15
    from structured_tasks import GUIDE, HELDOUT, LABELS, REFERENCE_SYSTEM, REFERENCE_USER # Session 16
    from structured_utils import SUPPORT_DIR, SYSTEM_FILE, USER_FILE, load_prompt_file, pick_examples  # Session 16
    from evaluation_tasks import (DEFAULT_FORMAT_RULE, GOLDEN20, JUDGE_SYSTEM, JUDGE_USER, RUBRIC, RUBRIC_IDS,  # today
                                  SAMPLE_REPLY_A, SAMPLE_REPLY_B, SAMPLE_TICKET, V1_INSTRUCTION)
    from evaluation_utils import (JudgeParseError, S17_CACHE, agreement, agreement_rate, answer_with,           # today
                                  check_s17_work, compare_prompts, escape_judge_tags, eval_markdown, judge_answer,
                                  judge_messages, parse_eval_report, parse_spot_check, parse_verdict, pass_rates,
                                  pct, save_eval_report, score_prompt, shuffle_order, spot_check_items, v1_messages,
                                  v2_messages_from_files)
except ImportError as e:
    raise ImportError(f"{e}. This notebook uses files from Sessions 11, 14, 15, 16 and today: run 'git pull' in the "
                      "course folder, then restart the kernel.") from e

MODEL = os.getenv("MODEL")
WORK_REPO = os.getenv("WORK_REPO")
cache, meter = DiskCache(S17_CACHE), Meter()
client = None
if os.getenv("BASE_URL") and os.getenv("API_KEY") and MODEL:
    client = groq_client()
lab = Lab(client, MODEL, cache=cache, meter=meter) if client else None     # one Lab plays BOTH roles today: answerer and judge
ready = lab is not None

print(f"model: {MODEL} | ready: {ready}")
print(f"answers are remembered in: {S17_CACHE}")
print(f"golden set: {len(GOLDEN20)} tickets · rubric: {len(RUBRIC)} criteria {RUBRIC_IDS}")
print(f"WORK_REPO: {WORK_REPO or '(not set — section 6 will need it)'}")
if not ready:
    print("No key: sections 4-6 print 'skipped'. Sections 2, 3, 7, 8 need no model and still run.")

### What you should see
```
model: openai/gpt-oss-20b | ready: True
answers are remembered in: C:\Users\you\.genai_course\s17_cache.json
golden set: 20 tickets · rubric: 4 criteria ['c1', 'c2', 'c3', 'c4']
WORK_REPO: C:\Users\you\Documents\genai-course-work
```
| If you see... | It means... | Do this |
|---|---|---|
| `ImportError ... run 'git pull'` | a course file is missing | `git pull`, `uv sync`, then **Kernel → Restart** |
| `ready: False` but you have a key | `.env` not found or a name is misspelled | `.env` needs `BASE_URL`, `API_KEY`, `MODEL`, in the course folder |
| `WORK_REPO: (not set...)` | section 6 will fail with a clear message | set `WORK_REPO` in `.env` or the shell, see Session 16 |

### Check yourself
*Today one `lab` plays two different roles in different calls — the "answerer" and the "judge". Why is it still fair to call these two SEPARATE calls, even though they may use the same underlying model?*

> **Common mistake:** forgetting that today's cache (`s17_cache.json`) is separate from Session 16's (`s16_cache.json`). If section 5's numbers look stale after you edited something, check you are not accidentally reading last session's cache.

## 2. Predict first: two replies, which is better?

### What and why
Before you build any machinery, try the actual judgement call yourself, on one ticket, with no tooling at all. This is the gap that today's rubric and judge exist to close in a repeatable way.

### The idea
A label is one bit. A reply is read on several axes: is the label right, does the reasoning make sense, does it follow the format support needs, does it promise something nobody approved. Two replies can both "get the label right" and still not be equally good.

### What the next cell does
1. Prints the ticket and both candidate replies, `SAMPLE_REPLY_A` and `SAMPLE_REPLY_B` — plain data, no model call.
2. You read them and form your own opinion before section 4 builds a judge.

### The code

In [ ]:
print("TICKET:", SAMPLE_TICKET)
print()
print("REPLY A:", SAMPLE_REPLY_A)
print()
print("REPLY B:", SAMPLE_REPLY_B)

### What you should see
Both replies, printed as written above (this needs no model and no key — it always runs).

**Which reply is better? A, B, neither, or both — differently?**

Look closely: reply A says "account issue" (the ticket is a UPI transfer — Session 16's house rules say UPI problems are `account`, so the LABEL is right). Reply B says "card" and invents a "24-hour cooling-off period" that appears nowhere in the bank's rules — the LABEL is wrong AND it states a policy nobody approved.

### Check yourself
*What exactly is wrong with reply B, beyond the label? Name at least two separate problems.*

> **Common mistake:** judging only the label and ignoring everything else the reply says. A classifier that gets the label right but invents a refund policy is not "basically fine" — in a real bank, an invented policy is a compliance incident.

## 3. The golden set and the rubric

### What and why
Two decisions, both made BEFORE any answer is read: which tickets to test on (the golden set), and what "good" means (the rubric). Deciding these after reading answers is grading toward whichever answer you already liked — exactly the leak-guard discipline Session 16 applied to few-shot examples, applied here to judging.

### The idea
`GOLDEN20` is not new data — look at the code below: it is literally `structured_tasks.HELDOUT`, the same object, same twenty tickets Session 16 measured label accuracy on. The name changes because the JOB changes (today it is graded on four axes, not one), not because the data changes.

### What the next cell does
1. `GOLDEN20 is HELDOUT` — Python's `is` checks they are literally the same object in memory, not just equal-looking copies.
2. Prints the four rubric criteria, each with its one-line test — read them now, before section 4 ever calls a judge.

### The code

In [ ]:
print("GOLDEN20 is Session 16's HELDOUT:", GOLDEN20 is HELDOUT)
print(f"{len(GOLDEN20)} tickets, 5 per label: {LABELS}")
print()
for c in RUBRIC:
    print(f"{c['id']}. {c['name']}")
    print(f"   {c['test']}")

### What you should see
```
GOLDEN20 is Session 16's HELDOUT: True
20 tickets, 5 per label: ['card', 'loan', 'account', 'other']

c1. Correct label
   The reply names exactly one of: card, loan, account, other - and it is the label this ticket is actually filed under...
c2. Cites the right house rule
   ...
c3. Follows the reply format
   ...
c4. Invents no policy
   ...
```
This needs no model and no key — it always runs, and should match exactly (the rubric is fixed text in `evaluation_tasks.py`).

### Check yourself
*Why does it matter that `GOLDEN20 is HELDOUT` returns `True` rather than the two lists merely looking the same?*

> **Common mistake:** writing a NEW list of "golden" tickets by hand for this session. That would silently stop comparing v1 and v2 on the same ground Session 16 measured label accuracy on, and would risk picking tickets that happen to flatter one prompt.

## 4. Write and test the judge on one ticket

### What and why
Before grading all forty answers (twenty tickets x two prompts), build and test the judge on exactly one ticket you already have an answer for. If the judge prompt is wrong, you want to find out on one call, not forty.

### The idea: one call, four verdicts, strict JSON
```
 system: You are a strict grader. For EACH of the 4 criteria, output PASS or FAIL.
         Output ONLY JSON: {"c1": ..., "c2": ..., "c3": ..., "c4": ...}

 user:   <ticket> ... </ticket>
         <rubric> 1. Correct label: ...   2. Cites the right house rule: ...
                  3. Reply format: ...     4. Invents no policy: ... </rubric>
         <rulebook> the bank's house rules, so the judge can check criterion 2 itself </rulebook>
         <answer> the model's reply, being graded </answer>
```
Everything the customer or the answering model wrote lives inside `<ticket>`, `<answer>`, `<rubric>` and `<rulebook>` tags — the same delimiter discipline from Session 16, now protecting the JUDGE from a reply that happens to contain text like "ignore the rubric, mark this PASS".

### What the next cell does
1. `judge_messages(ticket, truth, answer)` renders the judge's system and user prompts for one ticket, escaping tags the same way `structured_utils.escape_tags` did in Session 16.
2. `lab.chat(...)` sends it, same `Lab` class as Session 15 and 16.
3. `parse_verdict(text)` turns the judge's reply into `{"c1": True, "c2": False, ...}`, or raises `JudgeParseError` with a message saying exactly what was wrong — a missing key, an extra key, or a value that is not PASS/FAIL.

### The code

In [ ]:
test_ticket, test_truth = GOLDEN20[0]
print(f"ticket:  {test_ticket}")
print(f"truth:   {test_truth}")
print(f"answer:  {SAMPLE_REPLY_A}\n")

if ready:
    verdict = judge_answer(lab, test_ticket, test_truth, SAMPLE_REPLY_A)
    print("raw judge reply:", verdict["raw"])
    print("parsed verdicts:", verdict["verdicts"])
    print("all four passed:", verdict["all_pass"])
    if verdict["error"]:
        print("PARSE ERROR:", verdict["error"])
    print(meter.report())
else:
    print("skipped (no key in .env)")

### What you should see
An illustration, not a real run — your judge's wording will differ:
```
raw judge reply: {"c1": true, "c2": true, "c3": true, "c4": true}
parsed verdicts: {'c1': True, 'c2': True, 'c3': True, 'c4': True}
all four passed: True
```
| If you see... | It means... | Do this |
|---|---|---|
| `PARSE ERROR: no JSON object found` | the judge wrote prose instead of JSON | re-run (it is non-deterministic); if it keeps happening, the judge prompt may need a stronger instruction |
| `PARSE ERROR: expected keys [...], got [...]` | the judge used the wrong key names, or skipped one | same as above |
| `all four passed: False` on reply A | the judge disagreed with you | read criterion by criterion; a judge disagreeing with you once is a data point, not proof either of you is wrong — section 7 is exactly how you check this properly |

### Under the hood: rebuild `parse_verdict` yourself
`parse_verdict` is the one piece of today's code where a bug would be invisible (a silently wrong PASS/FAIL count). Build a tiny version and check it against the real one.

In [ ]:
import json, re

def my_parse_verdict(text, ids=("c1", "c2", "c3", "c4")):
    m = re.search(r"\{.*\}", text, re.DOTALL)
    if not m:
        raise ValueError("no JSON object found")
    obj = json.loads(m.group(0))
    out = {}
    for i in ids:
        if i not in obj:
            raise ValueError(f"missing key {i}")
        v = str(obj[i]).strip().upper()
        if v not in ("PASS", "FAIL"):
            raise ValueError(f"{i} is not PASS/FAIL: {obj[i]!r}")
        out[i] = v == "PASS"
    return out

sample = 'Sure, here you go:\n{"c1": "PASS", "c2": "FAIL", "c3": "PASS", "c4": "PASS"}\nhope that helps!'
mine, real = my_parse_verdict(sample), parse_verdict(sample)
print("mine:", mine)
print("real:", real)
assert mine == real == {"c1": True, "c2": False, "c3": True, "c4": True}
print("match: True")

### What you should see
```
mine: {'c1': True, 'c2': False, 'c3': True, 'c4': True}
real: {'c1': True, 'c2': False, 'c3': True, 'c4': True}
match: True
```
This cell needs no model and no key — it always runs.

### Check yourself
*Why does the real `parse_verdict` check for EXTRA keys too, not just missing ones? What could an extra key signal?*

> **Common mistake:** trusting a judge's free-text explanation ("looks mostly fine") as if it were a verdict. If you want an explanation, ask for it as a FIFTH key in the same JSON object, never as a replacement for the four PASS/FAIL keys.

## 5. Score v1 (baseline) on all 20

### What and why
Now the full run for ONE prompt version: Session 14's one-line baseline, answering all twenty golden tickets, each answer graded by the judge against all four criteria. This is the slow cell of the notebook — forty model calls (twenty answers, twenty judgements).

### The idea
`score_prompt` does exactly what you did by hand in sections 2 and 4, twenty times, and totals the result per criterion. Nothing new conceptually — this is scale, not a new idea.

### What the next cell does
1. `v1_messages(ticket)` builds the same messages Session 14 used: one system line, the ticket pasted in raw.
2. `score_prompt(lab, lab, v1_messages, GOLDEN20)` answers every ticket, judges every answer, and returns pass counts per criterion.
3. The same `lab` plays answerer and judge here — see the check-yourself question in section 1.

### The code

In [ ]:
if ready:
    v1_result = score_prompt(lab, lab, v1_messages, GOLDEN20)
    for c in RUBRIC:
        print(f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n'])}")
    print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n'])}")
    print(f"graded: {v1_result['graded']}/{v1_result['n']}")
    print(meter.report())
else:
    print("skipped (no key in .env)")
    v1_result = None

### What you should see
An illustration, not a real run:
```
Correct label                70%
Cites the right house rule   40%
Follows the reply format     35%
Invents no policy            90%
All four criteria            30%
graded: 20/20
```
Expect `Correct label` to roughly match Session 14's own number (same prompt, same tickets — a different criterion, graded by a judge rather than exact string matching, so small differences are normal). The other three criteria are typically much lower: a one-line instruction was never asked to cite a rule, follow a format, or avoid inventing anything.

| If you see... | It means... | Do this |
|---|---|---|
| `graded` less than 20 | the judge's reply could not be parsed for some tickets | normal occasionally; re-run if it is more than 2-3 tickets |
| very slow / pauses | the free plan's rate limit (about 8,000 tokens/minute) | wait; do not click Stop — forty calls take longer than Session 16's twenty |
| `All four criteria` is 0% | the baseline never does all four at once | expected — this IS the gap Session 16 and today exist to close |

### Check yourself
*Why is `All four criteria` almost always much lower than any single criterion on its own?*

> **Common mistake:** only looking at `Correct label` and concluding "the baseline is basically fine". The other three rows are exactly what a label-only accuracy number, like Session 14's and Session 16's, cannot see.

## 6. Score v2 (structured) on all 20, compare

### What and why
The actual comparison: v2 is loaded from YOUR Session 16 files, right now, from your work repository — not a copy stored in the course repo. If you have since edited `system.md`, today grades whatever is there today.

### The idea
`compare_prompts` runs v1 and v2 side by side, on the same twenty tickets, with the same judge — and applies the **order guard**: for each ticket, which prompt gets answered and judged first is decided by `shuffle_order`, deterministically, so neither prompt is always "first" (and so position bias, named in Part A, cannot systematically favour one side).

### What the next cell does
1. `v2_maker = lambda t: v2_messages_from_files(WORK_REPO, t)` builds a function that, for a ticket, loads your files **fresh** and renders the Session 16 template — `functools.partial` would also work; a `lambda` is simplest here.
2. `compare_prompts(lab, lab, v1_messages, v2_maker, GOLDEN20)` answers and judges every ticket for both prompts, order-guarded, and returns `{"v1": ..., "v2": ..., "order": [...]}`.
3. If `WORK_REPO` is not set, or your Session 16 files are missing, this raises `PromptFileError` with the exact fix — the same honest-failure rule as every session.

### The code

In [ ]:
if ready:
    if not WORK_REPO:
        print("skipped: WORK_REPO is not set. Add it to .env or set it in the shell, then restart the kernel.")
        cmp = None
    else:
        v2_maker = lambda t: v2_messages_from_files(WORK_REPO, t)
        cmp = compare_prompts(lab, lab, v1_messages, v2_maker, GOLDEN20)
        v1_result, v2_result = cmp["v1"], cmp["v2"]
        print(f"{'Criterion':<28} {'v1 (baseline)':<16} {'v2 (structured)':<16}")
        for c in RUBRIC:
            print(f"{c['name']:<28} {pct(v1_result['per_criterion'][c['id']], v1_result['n']):<16} "
                  f"{pct(v2_result['per_criterion'][c['id']], v2_result['n']):<16}")
        print(f"{'All four criteria':<28} {pct(v1_result['all_pass'], v1_result['n']):<16} "
              f"{pct(v2_result['all_pass'], v2_result['n']):<16}")
        print(f"\nticket order was v1-first for {sum(cmp['order'])}/{len(cmp['order'])} tickets (shuffled, not all-v1-first)")
        print(meter.report())
else:
    print("skipped (no key in .env)")
    cmp = None

### What you should see
An illustration, not a real run — your own numbers will differ, and should differ from your neighbour's:
```
Criterion                    v1 (baseline)    v2 (structured)
Correct label                70%              80%
Cites the right house rule   40%              90%
Follows the reply format     35%              95%
Invents no policy            90%              95%

All four criteria            30%              75%

ticket order was v1-first for 11/20 tickets (shuffled, not all-v1-first)
```
**The shape of the gap matters more than the exact numbers.** Expect v2 to lead most on "cites the right rule" and "follows the format" — that is precisely what Session 16's system prompt and template were built for. A roughly even `Correct label` gap, with a much bigger `All four criteria` gap, is the headline: Session 16's accuracy number alone would have told you far less than this table does.

| If you see... | It means... | Do this |
|---|---|---|
| `PromptFileError: prompt file not found` | `WORK_REPO` is wrong, or your Session 16 files are missing | check the path; your Session 16 lab must be done first |
| v2 is NOT better than v1 on most criteria | possible — read a few of v2's failing rows (see `cmp['v2']['rows']`) | this is real information about your own system.md, worth fixing before Session 18 |
| ticket order is lopsided (e.g. 19/20 v1-first) | `shuffle_order`'s hash happened to land that way for this particular 20 tickets | this is still deterministic and fair — a biased-looking split of a small set is not the same as a biased coin |

### Check yourself
*If `shuffle_order` were replaced with "always answer v1 first, then v2", and v2 then scored higher, could you be sure that was not partly a position-bias artifact? Why or why not?*

> **Common mistake:** reading only the `All four criteria` row and skipping the per-criterion table. The per-criterion breakdown is what tells you WHERE a prompt needs work, which the single number cannot.

## 7. Human spot-check (5 tickets)

### What and why
The judge is a model, with the biases Part A named. Before you trust its forty verdicts, grade a small, fixed sample yourself and see how often you agree. This is not proof the judge is right everywhere — it is evidence you have actually checked, which is the honest claim Session 17's checkpoint asks you to be able to make.

### The idea
`spot_check_items()` picks five tickets from the golden set the same deterministic way Session 16's `pick_examples` chose few-shot examples — same five every run, no hidden randomness, so your spot-check is reproducible and comparable across re-runs.

### What the next cell does
1. `spot_check_items()` returns five `(ticket, truth)` pairs.
2. For each, the cell prints the ticket, v2's answer (from section 6's `cmp`), and the judge's verdict.
3. **You fill in `MY_VERDICTS`** — your own PASS/FAIL per criterion for each of the five, read the rubric in section 3 again if you need to.
4. `agreement_rate(pairs)` compares your verdicts with the judge's, criterion by criterion (5 tickets x 4 criteria = 20 comparisons).

### The code

In [ ]:
spot = spot_check_items()
v2_rows_by_ticket = {r["ticket"]: r for r in cmp["v2"]["rows"]} if cmp else {}

for i, (ticket, truth) in enumerate(spot, 1):
    row = v2_rows_by_ticket.get(ticket)
    print(f"[{i}] truth={truth}")
    print(f"    ticket: {ticket}")
    if row:
        print(f"    v2 answer: {row['answer']}")
        print(f"    judge said: {row['verdicts']}")
    else:
        print("    (run section 6 first to have an answer + judge verdict to compare against)")
    print()

# YOUR TURN: grade these five yourself, True = PASS, False = FAIL, using the section 3 rubric
MY_VERDICTS = [
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 1 - edit these five dicts to your own judgement
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 2
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 3
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 4
    {"c1": True, "c2": True, "c3": True, "c4": True},    # ticket 5
]

if cmp:
    judge_verdicts = [v2_rows_by_ticket[t]["verdicts"] for t, _ in spot]
    pairs = list(zip(MY_VERDICTS, judge_verdicts))
    spot_result = agreement_rate(pairs)
    print("agreement with the judge:", spot_result)
else:
    spot_result = None
    print("skipped: run section 6 first")

### What you should see
Five tickets with v2's answer and the judge's verdict, then (after you edit `MY_VERDICTS` to your own honest reading):
```
agreement with the judge: {'agreed': 18, 'total': 20, 'rate': '18/20 = 90%'}
```
| If you see... | It means... | Do this |
|---|---|---|
| agreement below about 70% | you and the judge disagree a lot | read the disagreeing criteria closely — is the rubric ambiguous, or is the judge actually wrong? Either is a real finding |
| `KeyError` on `v2_rows_by_ticket` | section 6 was skipped or `cmp` is `None` | run section 6 first |

### Under the hood: rebuild `agreement` yourself

In [ ]:
def my_agreement(human, judge):
    ids = sorted(set(human) & set(judge))
    agree = [i for i in ids if human[i] == judge[i]]
    return {"ids": ids, "agree": agree, "disagree": [i for i in ids if i not in agree], "n": len(ids), "agreed": len(agree)}

h = {"c1": True, "c2": True, "c3": False, "c4": True}
j = {"c1": True, "c2": False, "c3": False, "c4": True}
mine, real = my_agreement(h, j), agreement(h, j)
print("mine:", mine)
print("real:", real)
assert mine == real == {"ids": ["c1", "c2", "c3", "c4"], "agree": ["c1", "c3", "c4"], "disagree": ["c2"], "n": 4, "agreed": 3}
print("match: True")

### What you should see
```
mine: {'ids': ['c1', 'c2', 'c3', 'c4'], 'agree': ['c1', 'c3', 'c4'], 'disagree': ['c2'], 'n': 4, 'agreed': 3}
real: {'ids': ['c1', 'c2', 'c3', 'c4'], 'agree': ['c1', 'c3', 'c4'], 'disagree': ['c2'], 'n': 4, 'agreed': 3}
match: True
```
This cell needs no model and no key — it always runs.

### Check yourself
*Why is agreement counted criterion by criterion (20 comparisons for 5 tickets) rather than ticket by ticket (5 comparisons)?*

> **Common mistake:** grading `MY_VERDICTS` AFTER reading what the judge said, then unconsciously agreeing with it. Form your own opinion on each ticket from the ticket and the answer alone, before you look at the judge's verdict.

## 8. Save your evaluation report, then check your whole submission

### What and why
Everything above lived only in this notebook's memory. `experiments/s17_eval.md` is the committed, reviewable record: pass rates for both prompts, and your spot-check agreement, in one file a teammate (or the checkpoint) can read without re-running anything.

### The idea
Same pattern as Session 16's `s16_heldout.md`: a markdown table plus a few summary lines, built by one function so the format is always the same, then saved to a fixed path the checkpoint knows to look for.

### What the next cell does
1. `eval_markdown(v1_result, v2_result, MODEL, MODEL, spot_check=spot_result)` builds the report text.
2. `save_eval_report(WORK_REPO, text)` writes it to `experiments/s17_eval.md` in your work repository.
3. `check_s17_work(WORK_REPO)` re-reads everything from disk (not from this notebook's memory) and runs the same plain-English checks the pytest checkpoint will — so you see problems here, before running pytest.

### The code

In [ ]:
SAVE_REPORT = True    # set False to preview without writing the file

if cmp and spot_result:
    report_text = eval_markdown(cmp["v1"], cmp["v2"], MODEL, MODEL, spot_check=spot_result)
    print(report_text)
    if SAVE_REPORT and WORK_REPO:
        path = save_eval_report(WORK_REPO, report_text)
        print(f"saved to {path}")
        print()
        for r in check_s17_work(WORK_REPO):
            print(f"  {'PASS' if r.passed else 'FAIL'}  {r.name}" + (f"   ({r.detail})" if r.detail else ""))
else:
    print("skipped: run sections 6 and 7 first")

### What you should see
The report text (a markdown table like section 6's, plus the spot-check line), then `saved to ...experiments\s17_eval.md`, then a list of checks that should all say `PASS`:
```
  PASS  work repo folder exists
  PASS  Session 16 prompt files are still there (system.md, user.j2)
  PASS  evaluation report exists
  PASS  report states pass rates for both prompts
  PASS  report covers all twenty tickets for both prompts
  PASS  human spot-check of at least 5 tickets is recorded
```
| If you see... | It means... | Do this |
|---|---|---|
| `FAIL  report covers all twenty tickets...` | section 6 ran on fewer than 20 tickets, or the judge could not parse several replies | check `cmp['v1']['graded']` and `cmp['v2']['graded']`; re-run section 6 |
| `FAIL  human spot-check...` | `spot_result` is `None`, or fewer than 5 tickets were recorded | redo section 7; make sure `MY_VERDICTS` has five entries |
| any `FAIL` on the prompt files | your Session 16 `system.md`/`user.j2` are missing from `WORK_REPO` | do not delete them; today grades them, it does not recreate them |

### Check yourself
*`check_s17_work` re-reads everything from the saved file on disk rather than from this notebook's `cmp` and `spot_result` variables. Why does that matter?*

> **Common mistake:** running section 8 once, getting all PASS, then re-running section 6 or 7 afterwards (new numbers) without re-running section 8. The saved report only reflects what it was saved from.

## 9. Commit, and run the checkpoint

### What and why
`experiments/s17_eval.md` goes to your WORK repository. The pytest checkpoint runs from the COURSE repository and needs `WORK_REPO` to find it — same split as every session since 14.

### What the next cell does
A shell block, not Python: commit the one new file, then run the checkpoint with `-rs` so a skip always prints its reason.

### The code

In [ ]:
# Run these in your terminal (PowerShell), not in this notebook:
#
# cd ~\Documents\genai-course-work
# git add experiments\s17_eval.md
# git commit -m "Session 17: evaluation report, v1 vs v2"
# git push
#
# cd ~\Documents\genai-agents-course
# $env:WORK_REPO = "$HOME\Documents\genai-course-work"
# uv run pytest tests\test_s17.py -rs
print("Copy the commands above into PowerShell. This cell is documentation, not something to run in the notebook.")

### What you should see
`git commit` reports one file changed; `pytest` prints `1 passed` (plus any earlier sessions' checkpoints you also ran) if `experiments/s17_eval.md` is complete, or the FAIL list from section 8 otherwise.

| If you see... | It means... | Do this |
|---|---|---|
| test SKIPPED | no key in `.env` | this is not a pass; add your key, see Session 11 |
| test FAILS with your check-list reasons | the report is incomplete | fix the specific FAIL from section 8, re-save, re-commit |

### Check yourself
*Your Session 16 `prompts/support_classifier/` files are never pushed again today. Why not — what already has them?*

> **Common mistake:** committing from the COURSE repository by mistake. `experiments/` lives in your WORK repository, same as `prompts/` has since Session 16.

---
# Part C · Wrap-up

## C1. Summary
- A label match is not the same as a good reply: a rubric of pass/fail criteria, decided before reading any answer, measures what accuracy alone cannot.
- The golden set is not new data — it is Session 16's held-out split, reused, so v1 and v2 are measured on ground neither has ever seen.
- An LLM judge returns strict JSON (four fixed keys, PASS/FAIL only), never a free-text score, so results can be totalled by code.
- The judge has known biases (position, verbosity, self-preference); shuffling answer order per ticket guards against the first.
- A human spot-check of five tickets is not proof the judge is right everywhere — it is evidence you checked, which is the honest claim the checkpoint lets you make.
- Hold everything fixed except the one thing you are comparing: same golden set, same rubric, same judge, same shuffle seed, for both v1 and v2.

## C2. The words again

| Term | One-line meaning |
|---|---|
| Golden set | a fixed, agreed test set, scored every version against |
| Rubric | pass/fail checks, decided before reading any answer |
| LLM-as-judge | a second model call grading the first model's output |
| Strict-JSON verdict | a fixed-shape machine-parseable score |
| Position bias | preferring whichever answer is read first |
| Verbosity bias | preferring longer, more confident replies |
| Self-preference bias | a model favouring its own style |
| Order guard | deterministically varying which prompt goes first, per item |
| Human spot-check | grading a sample yourself to check the judge |
| Pass rate | fraction of graded items passing a criterion |

## C3. Interview practice

**1. "How would you evaluate whether a new prompt is better than the old one?"**
Build a fixed golden set and a rubric of pass/fail criteria decided before reading any answers, then score both prompt versions on it with an LLM judge, same judge and same order-guard for both. Today's lab: Session 14's baseline against Session 16's structured prompt, on Session 16's own held-out tickets. Honest limit: a judge disagreeing with every human reviewer on a criterion is a real failure mode, which is exactly why section 7's spot-check exists.

**2. "What is LLM-as-judge, and what are its known failure modes?"**
Using a second model call to grade the first model's output against a rubric. Known biases: position (favours whichever answer is read first), verbosity (favours longer, more confident text), self-preference (favours its own phrasing style). Today's lab names all three and applies order-guarding for the first. Honest limit: this lab's answerer and judge share a model family, so self-preference bias is present and not mitigated here.

**3. "Why would you shuffle answer order in a judge call?"**
To stop a fixed, predictable ordering from systematically favouring one side regardless of quality. `shuffle_order()` decides, per ticket, which prompt is judged first, deterministically (same every run) but not fixed to one prompt always going first. Honest limit: with pass/fail-per-answer scoring (not true side-by-side pairwise comparison), this guards against a *subtler* timing/caching artifact rather than the textbook "sees both answers in one call" position bias.

**4. "What is the difference between a golden set and a held-out set?"**
Often none in the data itself — the difference is the job. "Held-out" means not used as a training or few-shot example (Session 16's sense). "Golden" means the fixed set every prompt version is scored against (today's sense). Today's `GOLDEN20` is literally Session 16's `HELDOUT`, reused.

**5. "Would you ever use a human to check an LLM judge, and how?"**
Yes — grade a small, fixed, representative sample yourself with the same rubric, and compute an agreement rate with the judge. Today: five tickets, four criteria each, twenty comparisons. Honest limit: five tickets is evidence on five tickets, not a guarantee for every ticket the judge will ever grade; a production system would re-check this periodically, not once.

## C4. Troubleshooting

| Error | Cause | Fix |
|---|---|---|
| `ImportError` on `evaluation_tasks` or `evaluation_utils` | course repo out of date | `git pull`, `uv sync`, restart kernel |
| `PromptFileError: prompt file not found` in section 6 | `WORK_REPO` wrong, or Session 16 files missing | check the path; do Session 16 first |
| `JudgeParseError` shown as `judge_error` on many rows | the judge's reply was not clean JSON | expected occasionally; a lot of them suggests the model is struggling — try again, it is not deterministic |
| checkpoint test SKIPPED | no key in `.env` | not a pass; add your key (Session 11) |
| checkpoint test FAILS with a FAIL list | `experiments/s17_eval.md` incomplete | fix the specific FAIL from section 8, re-save, re-commit |
| numbers differ from your neighbour's | models are not perfectly repeatable, and the judge is a model too | expected; read your own screen, not theirs |

## C5. What comes next
Session 18, **defensive prompting**, attacks the Session 16 prompt on purpose — five different ways — and re-measures it with this same judge and rubric, so you see directly whether your tags held up under attack, not just against ordinary tickets. Keep `prompts/support_classifier/`, `evaluation_tasks.py`'s rubric, and today's `experiments/s17_eval.md`: all three are reused, not rebuilt.